# Lab 2 — เขียน อัปเดต ลืม และทบทวน (EP12)

Lab 1 จบด้วยปัญหา: ความจำเรื่องเดียวกันที่ใช้คำต่างกัน หลุดการกันซ้ำ แล้วเบียดหัวข้อรายงานออกไป
Lab นี้ให้ **model ช่วยจัดการความจำ** ทั้งสี่งาน (§17.4)

**สิ่งที่จะได้ทำ**
1. **Write:** model ให้คะแนนความสำคัญ 1–10 → เก็บเฉพาะที่ถึงเกณฑ์ τ
2. **ดึงเป็นข้อเท็จจริงสั้นๆ** (atomic facts) แบบ Mem0 (§17.11.2)
3. **Update:** ADD / DUPLICATE / UPDATE — ข้อเท็จจริงเก่าไม่ถูกลบ แต่ถูกปิดด้วยเวลา `valid_until`
4. **Forget:** ลบเมื่อผู้ใช้ขอ (§17.5 privacy)
5. **Reflect:** ความล้มเหลวจริงสามครั้ง → บทเรียนหนึ่งข้อ → งานถัดไปผ่าน

คำตอบของ model ทุกคำตอบมาจากการรันจริง

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §17.4, §17.5, §17.11.2 — CC BY-SA 4.0

## 0. เตรียมเครื่องก่อนเริ่ม

| สิ่งที่ต้องมี | รายละเอียด |
|---|---|
| Python 3.10 ขึ้นไป | หรือใช้ Google Colab ซึ่งมีให้แล้ว |
| OpenRouter API key | สมัครที่ openrouter.ai → Keys → Create Key แล้วเติมเครดิต (lab นี้ใช้ไม่ถึง 0.1 เซ็นต์ดอลลาร์) |
| Packages | `openai`, `python-dotenv`, `numpy` (ติดตั้งใน cell ถัดไป) |

**ใส่ API key**
- **Colab:** ไอคอน 🔑 Secrets → ชื่อ `OPENROUTER_API_KEY` → เปิด *Notebook access*
- **เครื่องตัวเอง:** สร้างไฟล์ `.env` ในโฟลเดอร์เดียวกับ notebook แล้วใส่บรรทัด `OPENROUTER_API_KEY=sk-or-...`

⚠️ อย่าใส่ key ลงในโค้ดโดยตรง และอย่าแชร์ไฟล์ `.env`

In [ ]:
%pip install -q "openai>=1.50" python-dotenv numpy

## 1. Model และฟังก์ชันช่วย

Gemma 4 26B (Makora) ตอบ · Qwen3 Embedding 8B (Nebius) ทำ embedding — เหมือน Lab 1 (เหตุผลที่เลือก model อยู่ใน Lab 1)
ใหม่: `ask_json` — ขอคำตอบเป็น JSON ตาม schema (structured output จาก EP02)

In [ ]:
import os, json, re
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import numpy as np
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0, "embed": 0.0}

def chat(messages):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=messages)
    COST["chat"] += r.usage.cost
    return r.choices[0].message.content

def embed(text):
    r = client.embeddings.create(model=EMBED_MODEL, input=[text], extra_body=EMBED_EXTRA)
    COST["embed"] += r.usage.cost
    v = np.array(r.data[0].embedding)
    return v / np.linalg.norm(v)

In [ ]:
def ask_json(prompt, schema):
    r = client.chat.completions.create(
        model=MODEL, temperature=0, extra_body=EXTRA,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "reply", "strict": True, "schema": schema}})
    COST["chat"] += r.usage.cost
    return json.loads(r.choices[0].message.content)

## 2. คลังความจำ + `valid_until`

`MemoryEntry` ของ Lab 1 เพิ่มหนึ่งช่อง: **`valid_until`** — ข้อเท็จจริงที่ถูกแทนที่ ไม่ถูกลบ แต่ถูกปิดด้วยเวลา
("เก็บทั้งคู่พร้อมเวลา" — หนึ่งในสามทางเลือกของตำราเมื่อความจำขัดกัน, §17.4.1)
ค้นแบบย่อ: ความเหมือนอย่างเดียว เฉพาะความจำที่ยังจริงอยู่

In [ ]:
@dataclass
class MemoryEntry:
    content: str
    embedding: np.ndarray
    timestamp: datetime = field(default_factory=datetime.now)
    importance: float = 0.5
    valid_until: datetime | None = None          # set when a newer fact replaces this one

class MemoryStore:
    def __init__(self):
        self.entries = []

    def current(self):
        return [e for e in self.entries if e.valid_until is None]

    def search(self, q, k=3, everything=False):
        pool = self.entries if everything else self.current()
        if not pool:
            return []
        sims = np.stack([e.embedding for e in pool]) @ q
        return [(sims[i], pool[i]) for i in np.argsort(-sims)[:k]]

mem = MemoryStore()

## 3. Write: model ให้คะแนนความสำคัญ

`Write(e) = 1[importance(e) > τ]` (§17.4.1) — ใช้ "LLM self-assessment" ให้คะแนน 1–10 · เก็บถ้าได้ ≥ τ = 5

In [ ]:
IMPORTANCE = {"type": "object", "additionalProperties": False, "required": ["reason", "importance"],
              "properties": {"reason": {"type": "string"},
                             "importance": {"type": "integer", "minimum": 1, "maximum": 10}}}

def rate(message):
    return ask_json("You manage the long-term memory of a research assistant. "
                    "Facts about the user's work, deadlines and preferences matter in future sessions; "
                    "small talk and personal details unrelated to the work do not. "
                    "Rate from 1 to 10 how important it is to remember this user message "
                    "(10 = essential, 1 = small talk).\nMessage: " + message,
                    IMPORTANCE)["importance"]

## 4. ดึงข้อเท็จจริง แล้วตัดสินว่า ADD / DUPLICATE / UPDATE

แบบ Mem0 (§17.11.2): ดึงข้อเท็จจริงจากบทสนทนาเอง แล้วเทียบกับความจำที่ใกล้ที่สุด 3 ชิ้น
| action | ทำอะไร |
|---|---|
| ADD | ข้อมูลใหม่ → เก็บ |
| DUPLICATE | มีอยู่แล้ว → ไม่เก็บ (แก้ปัญหาของ Lab 1) |
| UPDATE | แทนของเก่าที่ไม่จริงแล้ว → ปิดของเก่าด้วย `valid_until` แล้วเก็บของใหม่ |

In [ ]:
FACTS = {"type": "object", "additionalProperties": False, "required": ["facts"],
         "properties": {"facts": {"type": "array", "items": {"type": "string"}}}}

def extract(message):
    return ask_json("Extract the facts about the user worth remembering from this message, "
                    "as short standalone sentences that start with 'The user'. "
                    "One fact per sentence.\nMessage: " + message, FACTS)["facts"]

DECISION = {"type": "object", "additionalProperties": False, "required": ["action", "target"],
            "properties": {"action": {"type": "string", "enum": ["ADD", "DUPLICATE", "UPDATE"]},
                           "target": {"type": "integer"}}}

def decide(fact, candidates):
    listing = "\n".join(f"{i}. {e.content}" for i, (_, e) in enumerate(candidates)) or "(none)"
    return ask_json(f"New fact about the user: {fact}\nExisting memories:\n{listing}\n"
                    "Choose one action:\n"
                    "ADD - the new fact is new information.\n"
                    "DUPLICATE - an existing memory already says the same thing.\n"
                    "UPDATE - an existing memory gives a different value for the same detail "
                    "(for example an old deadline and a new deadline), so it is no longer true.\n"
                    "Set target to the number of that existing memory, or -1 for ADD.", DECISION)

In [ ]:
def remember(message, when=None, tau=5):
    score = rate(message)
    print(f"[{score:>2}] {message}")
    if score < tau:
        return                                         # not worth remembering
    for fact in extract(message):
        emb = embed(fact)
        candidates = mem.search(emb)
        d = decide(fact, candidates)
        old = candidates[d["target"]][1] if 0 <= d["target"] < len(candidates) else None
        print(f"     {d['action']:<9} {fact}" + (f"  ← {old.content}" if old else ""))
        if d["action"] == "DUPLICATE":
            continue
        if d["action"] == "UPDATE" and old:
            old.valid_until = when or datetime.now()  # keep it, mark it as past
        mem.entries.append(MemoryEntry(fact, emb, when or datetime.now(), score / 10))

## 5. วันจันทร์ → วันอังคาร

ข้อความหกข้อของวันจันทร์ (เวลาจำลอง: เมื่อวาน) แล้วข้อความใหม่ของวันอังคาร

In [ ]:
yesterday = datetime.now() - timedelta(days=1)
monday = ["Hi! I'm writing a report on transformers for time series.",
          "Please keep your answers short.",
          "Thanks!",
          "By the way, I live in Bangkok.",
          "My report deadline is Friday.",
          "Short answers, please. Bullet points are fine."]
for message in monday:
    remember(message, when=yesterday)

In [ ]:
remember("My report deadline moved to next Wednesday.")
print()
for e in mem.entries:
    until = e.valid_until.strftime("%a %d %b") if e.valid_until else "now"
    print(f"{e.timestamp:%a %d %b} → {until:<10}  {e.importance:.1f}  {e.content}")

## 6. ใช้ความจำตอบ

In [ ]:
def answer(question, k=3):
    hits = mem.search(embed(question), k)
    for sim, e in hits:
        print(f"{sim:.2f}  {e.content}")
    system = "What you remember about this user (follow their preferences):\n" + \
             "\n".join(f"- {e.content}" for _, e in hits)
    return chat([{"role": "system", "content": system}, {"role": "user", "content": question}])

print(answer("When is my report due?"))

In [ ]:
print(answer("Any new papers for my report?")[:500])

## 7. Forget: ลบเมื่อผู้ใช้ขอ

ตำรา (§17.5 Privacy): ผู้ใช้ต้อง **ดูและลบ** ความจำของตัวเองได้ — ลบทั้งของปัจจุบัน **และประวัติ**

In [ ]:
FORGET = {"type": "object", "additionalProperties": False, "required": ["delete"],
          "properties": {"delete": {"type": "array", "items": {"type": "integer"}}}}

def forget(request):
    candidates = mem.search(embed(request), k=5, everything=True)
    listing = "\n".join(f"{i}. {e.content}" for i, (_, e) in enumerate(candidates))
    out = ask_json(f"The user asks: {request}\nMemories:\n{listing}\n"
                   "Which memories must be deleted to respect this request? Give their numbers.", FORGET)
    for i in out["delete"]:
        print("deleted:", candidates[i][1].content)
        mem.entries.remove(candidates[i][1])

forget("Please forget everything about my deadline.")

## 8. Reflect: จากความล้มเหลว สู่บทเรียน (§17.4.4)

ให้ model เขียนฟังก์ชันสามข้อ แล้วทดสอบกับ list ว่าง → เก็บผลเป็น **episodic memory**
→ ทบทวนเป็นบทเรียนหนึ่งข้อ (**semantic memory**) → งานใหม่ ไม่มี / มี บทเรียน

⚠️ cell นี้รันโค้ดที่ model เขียน (`exec`) บนเครื่องเราเอง — ทำได้ในการทดลองนี้เท่านั้น ระบบจริงต้องรันใน sandbox

In [ ]:
def try_empty(code, fn):
    ns = {}
    try:
        exec(code, ns)
        return f"ok → {ns[fn]([])!r}"
    except Exception as e:
        return f"{type(e).__name__}: {e}"

def attempt(fn, desc, lessons=()):
    messages = [{"role": "user", "content": f"Write a Python function {fn}(xs) that {desc}. Code only."}]
    if lessons:
        messages.insert(0, {"role": "system", "content": "Lessons from your past attempts:\n" +
                            "\n".join("- " + l for l in lessons)})
    code = re.search(r"```(?:python)?\n(.*?)```", chat(messages), re.S).group(1)
    return try_empty(code, fn)

In [ ]:
episodes = []
for fn, desc in [("mean", "returns the mean of a list of numbers"),
                 ("first_last", "returns a tuple of the first and last items of a list"),
                 ("range_of", "returns max minus min of a list of numbers")]:
    episodes.append(f"Task: write {fn}(xs). Test {fn}([]): {attempt(fn, desc)}")
    print(episodes[-1])

In [ ]:
INSIGHT = {"type": "object", "additionalProperties": False, "required": ["insight"],
           "properties": {"insight": {"type": "string"}}}

def reflect(episodes):
    return ask_json("These are past attempts of a coding agent:\n" +
                    "\n".join("- " + e for e in episodes) +
                    "\nWhat one general lesson should the agent remember for future tasks? "
                    "One sentence.", INSIGHT)["insight"]

insight = "[INSIGHT] " + reflect(episodes)
mem.entries.append(MemoryEntry(insight, embed(insight), importance=0.9))
print(insight)

In [ ]:
task = ("largest", "returns the largest number in a list")
hits = mem.search(embed(f"Write a Python function {task[0]}(xs) that {task[1]}."), k=1)
print(f"retrieved: {hits[0][1].content}\n")
for run in range(3):
    print("no memory   ", attempt(*task))
    print("with lesson ", attempt(*task, lessons=[hits[0][1].content]))

In [ ]:
print({k: f"${v:.5f}" for k, v in COST.items()})

## สรุป

| งาน | วิธี | สิ่งที่เห็น |
|---|---|---|
| Write | model ให้คะแนน 1–10, τ = 5 | "Thanks!" 1 · "I live in Bangkok" 1 → ไม่จำ · กำหนดส่ง 10 |
| Extract | atomic facts แบบ Mem0 | ข้อความเดียว → สองข้อเท็จจริง |
| Update | ADD / DUPLICATE / UPDATE | คำขอคำตอบสั้นครั้งที่สอง → DUPLICATE · วันพุธ → UPDATE วันศุกร์ (เก็บไว้เป็นประวัติ) |
| Read | ค้นเฉพาะความจำที่ยังจริง | "When is my report due?" → วันพุธหน้า · หัวข้อรายงานกลับมาในสามชิ้นแรก |
| Forget | ลบเมื่อผู้ใช้ขอ รวมประวัติ | ลบทั้งวันพุธและวันศุกร์ |
| Reflect | episodic → insight → semantic | `largest([])` ไม่มีบทเรียน: พัง 3/3 · มีบทเรียน: ผ่าน 3/3 |

**แบบฝึกหัด**
1. ตั้ง `tau=8` — ข้อความไหนหายไปบ้าง แล้ววันอังคารยังตอบได้ไหม
2. ลบประโยคเกณฑ์ ("Facts about the user's work…") ออกจาก `rate` แล้วรันซ้ำหลายรอบ — คะแนนของกำหนดส่งนิ่งไหม
3. เขียน `history(question)` ที่ค้นด้วย `everything=True` แล้วตอบว่า "เดิมรายงานต้องส่งเมื่อไร" (ทำก่อน `forget`)

**ถัดไป — Lab 3:** agent ที่มีความจำสามชั้น (hot / warm / cold) และเรียกความจำเองผ่าน tool — แล้ววัดผล